In [2]:
import os
import zipfile
import shutil
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# ==========================================
# 1. IMPOSTAZIONI E DECOMPRESSIONE
# ==========================================
zip_path = 'dataset 2.zip'  # Carica il tuo zip su Colab con questo nome
extract_path = './dataset_micro_completo'

if os.path.exists(zip_path):
    print("Decompressione del dataset in corso...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Decompressione completata!")
else:
    print(f"ATTENZIONE: Carica il file '{zip_path}' nella barra laterale di Colab.")

# ==========================================
# 2. PULIZIA DEI FILE CORROTTI
# ==========================================
print("\nVerifica e pulizia delle immagini corrotte...")
immagini_rimosse = 0
immagini_valide = 0

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            file_path = os.path.join(root, file)
            try:
                # Tentiamo di aprire e verificare l'integrità dell'immagine
                with Image.open(file_path) as img:
                    img.verify()
                # Nota: verify() non legge tutti i pixel, riapriamo per sicurezza
                with Image.open(file_path) as img:
                    img.resize((10, 10)) # Forza il caricamento effettivo dei dati
                immagini_valide += 1
            except (IOError, SyntaxError, Image.DecompressionBombError) as e:
                print(f"File corrotto rimosso: {file_path} (Errore: {e})")
                os.remove(file_path)
                immagini_rimosse += 1

print(f"Pulizia terminata. Immagini valide: {immagini_valide}, Rimosse perché corrotte: {immagini_rimosse}")

# ==========================================
# 3. ORGANIZZAZIONE DATASET (3 CLASSI)
# ==========================================
base_dir = './data_split'
classi = ['pericolo', 'non_pericolo', 'pentola']

# Reset cartelle precedenti se esistono
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

for split in ['train', 'val']:
    for categoria in classi:
        os.makedirs(os.path.join(base_dir, split, categoria), exist_ok=True)

# Smistamento in base ai prefissi dei file
all_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_files.append(os.path.join(root, file))

file_divisi = {
    'pericolo': [f for f in all_files if 'pericolo' in os.path.basename(f) and 'non' not in os.path.basename(f)],
    'non_pericolo': [f for f in all_files if 'non_pericolo' in os.path.basename(f)],
    'pentola': [f for f in all_files if 'pentola' in os.path.basename(f)]
}

for cat in classi:
    files_cat = file_divisi[cat]
    print(f"Classe '{cat}': trovate {len(files_cat)} immagini.")
    if len(files_cat) > 0:
        train_f, val_f = train_test_split(files_cat, test_size=0.2, random_state=42)
        for f in train_f:
            shutil.copy(f, os.path.join(base_dir, 'train', cat))
        for f in val_f:
            shutil.copy(f, os.path.join(base_dir, 'val', cat))

# ==========================================
# 4. CARICAMENTO DATASET IN TENSORFLOW
# ==========================================
IMG_SIZE = (128, 128)
BATCH_SIZE = 8

print("\nCaricamento dei dataset in TensorFlow...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base_dir, 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical' # Cambiato a categorical per 3 classi
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base_dir, 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# ==========================================
# 5. CREAZIONE E ADDESTRAMENTO RETE NEURALE
# ==========================================
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax') # 3 uscite con softmax per classificazione a 3 categorie
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy', # Ottimizzata per multiclasse
    metrics=['accuracy']
)

model.summary()

print("\nInizio addestramento del modello a 3 classi...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

# Salva il modello finale
model.save('modello_3_classi.h5')
print("\nModello salvato con successo come 'modello_3_classi.h5'")

Decompressione del dataset in corso...


BadZipFile: File is not a zip file

In [4]:
import os
import zipfile
import shutil

zip_path = 'dataset 2.zip'  # <--- ASSICURATI CHE IL NOME SIA IDENTICO A QUELLO CARICATO
extract_path = './dataset_micro'

# Controllo di sicurezza prima di iniziare
if os.path.exists(zip_path):
    file_size = os.path.getsize(zip_path) / (1024 * 1024) # Dimensione in MB
    print(f"File trovato: {zip_path} | Dimensione attuale: {file_size:.2f} MB")

    if file_size == 0:
        print("ATTENZIONE: Il file ha una dimensione di 0 MB! Il caricamento su Colab non è ancora finito o è fallito.")
    else:
        # Pulizia della vecchia cartella se presente
        if os.path.exists(extract_path):
            shutil.rmtree(extract_path)

        print("Tentativo di estrazione con metodo standard...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
            print("Decompressione completata con successo tramite Python!")
        except Exception as e:
            print(f"Errore con il modulo zipfile ({e}). Provo il metodo di emergenza Bash...")
            # Metodo di emergenza: usa l'unzip di Linux che ripara i piccoli errori di intestazione dello zip
            !unzip -q {zip_path} -d {extract_path}
            print("Decompressione forzata completata!")
else:
    print(f"ERRORE CRITICO: Il file '{zip_path}' non esiste nella cartella principale di Colab.")
    print("Verifica nella barra laterale di non averlo caricato dentro una sottocartella (es. 'sample_data').")

File trovato: dataset 2.zip | Dimensione attuale: 6.00 MB
Tentativo di estrazione con metodo standard...
Errore con il modulo zipfile (File is not a zip file). Provo il metodo di emergenza Bash...
unzip:  cannot find or open dataset, dataset.zip or dataset.ZIP.
Decompressione forzata completata!


In [5]:
import os
import numpy as np
from sklearn.model_init import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# 1. Configurazione dei percorsi in Colab
# Sostituisci 'cartella_pentola' con il nome esatto della cartella estratta se diverso
DATASET_DIR = "cartella_pentola"
IMG_SIZE = (240, 240) # Risoluzione ideale per i modelli Edge AI di OpenMV RT1062

X = [] # Conterrà i dati delle immagini
y = [] # Conterrà le etichette (visto che stiamo caricando la classe 'pentola')

print("Caricamento delle immagini in corso...")

# Elenca i file estratti nella cartella
for file in os.listdir(DATASET_DIR):
    if file.lower().endswith(('.jpg', '.jpeg', '.png')) and not file.startswith('.'):
        img_path = os.path.join(DATASET_DIR, file)
        try:
            # Carica l'immagine e la ridimensiona per il microcontrollore
            img = load_img(img_path, target_size=IMG_SIZE)
            img_array = img_to_array(img) / 255.0 # Normalizzazione dei pixel tra 0 e 1
            X.append(img_array)
            y.append(0) # Assegniamo classe 0 (es. 'pentola_acciaio')
        except Exception as e:
            print(f"Salto il file corrotto {file}: {e}")

X = np.array(X)
y = np.array(y)

print(f"Caricate con successo {len(X)} immagini.")

# 2. SPLITTING DEL DATASET (80% Train, 10% Validation, 10% Test)
print("\nEsecuzione dello splitting dei dati...")

# Primo split: separa il Train (80%) dal resto (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True
)

# Secondo split: divide il 20% rimanente a metà -> 10% Validation e 10% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, shuffle=True
)

print(f"--- Risultato dello Split ---")
print(f"Immagini per il TRAIN (Addestramento): {X_train.shape[0]}")
print(f"Immagini per la VALIDATION (Controllo): {X_val.shape[0]}")
print(f"Immagini per il TEST (Verifica finale): {X_test.shape[0]}")

ModuleNotFoundError: No module named 'sklearn.model_init'

In [6]:
import os
import zipfile
import shutil
from PIL import Image
# IMPORT CORRETTO: model_selection al posto di model_init
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# ==========================================
# 1. IMPOSTAZIONI E DECOMPRESSIONE DI SICUREZZA
# ==========================================
zip_path = 'dataset.zip'  # Carica il tuo zip su Colab con questo nome
extract_path = './dataset_micro'

if os.path.exists(zip_path):
    file_size = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"File trovato: {zip_path} | Dimensione: {file_size:.2f} MB")

    if file_size == 0:
        print("ATTENZIONE: Il file è vuoto. Attendi il completamento del caricamento.")
    else:
        if os.path.exists(extract_path):
            shutil.rmtree(extract_path)

        print("Decompressione del dataset in corso...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
            print("Decompressione completata con successo tramite Python!")
        except Exception as e:
            print(f"Errore con zipfile ({e}). Provo il metodo di emergenza Bash...")
            # Usa il comando di sistema se Python fallisce l'intestazione dello zip
            !unzip -q {zip_path} -d {extract_path}
            print("Decompressione di emergenza completata!")
else:
    print(f"ERRORE: File '{zip_path}' non trovato. Caricalo nella barra laterale di Colab.")

# ==========================================
# 2. PULIZIA AUTOMATICA DEI FILE CORROTTI
# ==========================================
print("\nVerifica e pulizia delle immagini corrotte...")
immagini_rimosse = 0
immagini_valide = 0

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            file_path = os.path.join(root, file)
            try:
                # Forza il caricamento effettivo dei dati dell'immagine per stanare file corrotti
                with Image.open(file_path) as img:
                    img.verify()
                with Image.open(file_path) as img:
                    img.resize((10, 10))
                immagini_valide += 1
            except Exception as e:
                print(f"File corrotto rimosso: {file_path}")
                os.remove(file_path)
                immagini_rimosse += 1

print(f"Pulizia terminata. Immagini valide: {immagini_valide}, Rimosse: {immagini_rimosse}")

# ==========================================
# 3. ORGANIZZAZIONE DATASET (3 CLASSI)
# ==========================================
base_dir = './data_split'
classi = ['pericolo', 'non_pericolo', 'pentola']

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

for split in ['train', 'val']:
    for categoria in classi:
        os.makedirs(os.path.join(base_dir, split, categoria), exist_ok=True)

# Recupera tutte le immagini estratte e pulite
all_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_files.append(os.path.join(root, file))

# Riconoscimento delle classi in base al nome dei file
file_divisi = {
    'pericolo': [f for f in all_files if 'pericolo' in os.path.basename(f) and 'non' not in os.path.basename(f)],
    'non_pericolo': [f for f in all_files if 'non_pericolo' in os.path.basename(f)],
    'pentola': [f for f in all_files if 'pentola' in os.path.basename(f)]
}

# Divisione in Train (80%) e Validation (20%)
for cat in classi:
    files_cat = file_divisi[cat]
    print(f"Classe '{cat}': trovate {len(files_cat)} immagini.")
    if len(files_cat) > 0:
        train_f, val_f = train_test_split(files_cat, test_size=0.2, random_state=42)
        for f in train_f:
            shutil.copy(f, os.path.join(base_dir, 'train', cat))
        for f in val_f:
            shutil.copy(f, os.path.join(base_dir, 'val', cat))

# ==========================================
# 4. PREPARAZIONE DATASET TENSORFLOW
# ==========================================
IMG_SIZE = (128, 128)
BATCH_SIZE = 8

print("\nCaricamento dei dataset in TensorFlow...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base_dir, 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(base_dir, 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# ==========================================
# 5. RETE NEURALE E ADDESTRAMENTO
# ==========================================
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')  # 3 uscite con attivazione softmax
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Perdita per problemi multi-classe
    metrics=['accuracy']
)

model.summary()

print("\nInizio addestramento del modello a 3 classi...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

# Salva il modello definitivo in formato Keras
model.save('modello_3_classi.h5')
print("\n[OK] Modello salvato come 'modello_3_classi.h5'")

ERRORE: File 'dataset.zip' non trovato. Caricalo nella barra laterale di Colab.

Verifica e pulizia delle immagini corrotte...
Pulizia terminata. Immagini valide: 0, Rimosse: 0
Classe 'pericolo': trovate 0 immagini.
Classe 'non_pericolo': trovate 0 immagini.
Classe 'pentola': trovate 0 immagini.

Caricamento dei dataset in TensorFlow...
Found 0 files belonging to 3 classes.


ValueError: No images found in directory ./data_split/train. Allowed formats: ('.bmp', '.gif', '.jpeg', '.jpg', '.png')

In [8]:
# ==========================================
# 3. ORGANIZZAZIONE DATASET (CORRETTO PER NO_PERICOLO)
# ==========================================
base_dir = './data_split'
classi = ['pericolo', 'no_pericolo', 'pentola'] # Le cartelle di output per TF

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

for split in ['train', 'val']:
    for categoria in classi:
        os.makedirs(os.path.join(base_dir, split, categoria), exist_ok=True)

# Recupera tutte le immagini estratte e pulite
all_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_files.append(os.path.join(root, file))

# Contenitori per lo smistamento
file_divisi = {'pericolo': [], 'non_pericolo': [], 'pentola': []}

for f in all_files:
    nome_file = os.path.basename(f).lower()
    percorso_completo = f.lower()

    # CORREZIONE: Controlla "no_pericolo" e lo assegna alla classe "non_pericolo" de modello
    if 'no_pericolo' in nome_file or 'no_pericolo' in percorso_completo:
        file_divisi['non_pericolo'].append(f)
    # Il controllo del pericolo generico esclude i file "no_pericolo"
    elif 'pericolo' in nome_file or 'pericolo' in percorso_completo:
        file_divisi['pericolo'].append(f)
    elif 'pentola' in nome_file or 'pentola' in percorso_completo:
        file_divisi['pentola'].append(f)

# Resoconto a schermo
print("\n--- RESOCONTO FILE TROVATI ---")
for cat in classi:
    print(f"Classe '{cat}': {len(file_divisi[cat])} immagini individuate.")
print("-------------------------------\n")

# Divisione in Train (80%) e Validation (20%)
for cat in classi:
    files_cat = file_divisi[cat]
    if len(files_cat) > 0:
        if len(files_cat) < 2:
            print(f"⚠️ Troppo poche immagini per la classe {cat}. Copio in train/val.")
            for f in files_cat:
                shutil.copy(f, os.path.join(base_dir, 'train', cat))
                shutil.copy(f, os.path.join(base_dir, 'val', cat))
        else:
            train_f, val_f = train_test_split(files_cat, test_size=0.2, random_state=42)
            for f in train_f:
                shutil.copy(f, os.path.join(base_dir, 'train', cat))
            for f in val_f:
                shutil.copy(f, os.path.join(base_dir, 'val', cat))


--- RESOCONTO FILE TROVATI ---
Classe 'pericolo': 0 immagini individuate.


KeyError: 'no_pericolo'

In [9]:
# ==========================================
# 1. CARICAMENTO DATASET A 3 CLASSI
# ==========================================
# TensorFlow leggerà in automatico 3 classi se nella cartella 'cartella_estratta'
# sono presenti 3 sottocartelle distinte (es. 'no_pericolo', 'pericolo', 'pentola')

IMG_SIZE = (64, 64)  # Mantenuto a 64x64 come nel tuo screenshot
BATCH_SIZE = 32

print("Caricamento dataset di addestramento...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/cartella_estratta',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'  # MODIFICATO: 'categorical' per gestire 3 classi
)

print("\nCaricamento dataset di validazione...")
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/cartella_estratta',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'  # MODIFICATO: 'categorical' per gestire 3 classi
)

# Stampa di verifica delle classi rilevate
class_names = train_ds.class_names
print(f"\n[OK] Classi rilevate da TensorFlow: {class_names}")


# ==========================================
# 2. DEFINIZIONE DEL MODELLO A 3 CLASSI
# ==========================================
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    # MODIFICATO: 3 uscite finali e attivazione 'softmax' per classificazione multi-classe
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # MODIFICATO: perdita adatta a 3 classi
    metrics=['accuracy']
)

model.summary()

Caricamento dataset di addestramento...


FileNotFoundError: [Errno 2] No such file or directory: '/content/cartella_estratta'

In [11]:
import os
import zipfile
import shutil

# Definiamo i percorsi (gestendo lo spazio nel nome del file)
zip_path = 'dataset 2.zip'
extract_path = '/content/cartella_estratta'

if os.path.exists(zip_path):
    file_size = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"File trovato: {zip_path} | Dimensione: {file_size:.2f} MB")

    # Pulizia della vecchia cartella se già esistente per evitare sovrascritture parziali
    if os.path.exists(extract_path):
        shutil.rmtree(extract_path)

    print(f"Decompressione di '{zip_path}' in corso...")
    try:
        # Metodo standard Python
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Decompressione completata con successo tramite Python!")
    except Exception as e:
        print(f"Nota con zipfile ({e}). Provo il metodo di emergenza Bash...")
        # Metodo Bash: le virgolette permettono a Linux di leggere correttamente lo spazio nel nome
        !unzip -q "dataset 2.zip" -d "{extract_path}"
        print("Decompressione di emergenza completata!")

    # Verifica del contenuto estratto
    classi_rilevate = [d for d in os.listdir(extract_path) if os.path.isdir(os.path.join(extract_path, d))]
    print(f"\n[OK] Estrazione completata in: {extract_path}")
    print(f"Sottocartelle (classi) create: {classi_rilevate}")

else:
    print(f"❌ ERRORE CRITICO: Il file '{zip_path}' non è stato trovato nella cartella principale di Colab.")
    print("Controlla nella barra laterale di non averlo rinominato o inserito per sbaglio in una sottocartella.")

File trovato: dataset 2.zip | Dimensione: 74.81 MB
Decompressione di 'dataset 2.zip' in corso...
Decompressione completata con successo tramite Python!

[OK] Estrazione completata in: /content/cartella_estratta
Sottocartelle (classi) create: ['__MACOSX', 'dataset']


In [13]:
import shutil
import os

# Percorsi attuali
cartella_estratta = '/content/cartella_estratta'
path_macosx = os.path.join(cartella_estratta, '__MACOSX')
path_dataset_interno = os.path.join(cartella_estratta, 'dataset')

# 1. RIMOZIONE DELLA CARTELLA MACOSX
if os.path.exists(path_macosx):
    shutil.rmtree(path_macosx)
    print("[OK] Cartella inutile __MACOSX eliminata con successo!")
else:
    print("Cartella __MACOSX non trovata o già eliminata.")

# 2. SISTEMAZIONE DELLA STRUTTURA PER TENSORFLOW
# Spostiamo il contenuto di 'dataset' direttamente in 'cartella_estratta'
if os.path.exists(path_dataset_interno):
    print("Spostamento delle classi nella cartella principale...")
    for elemento in os.listdir(path_dataset_interno):
        shutil.move(os.path.join(path_dataset_interno, elemento), cartella_estratta)
    # Rimuoviamo la cartella 'dataset' ora che è vuota
    os.rmdir(path_dataset_interno)
    print("[OK] File riorganizzati correttamente!")

# Verifica finale del risultato
classi_finali = [d for d in os.listdir(cartella_estratta) if os.path.isdir(os.path.join(cartella_estratta, d))]
print(f"\nStruttura finale pulita per l'allenamento. Classi reali pronte: {classi_finali}")

[OK] Cartella inutile __MACOSX eliminata con successo!
Spostamento delle classi nella cartella principale...
[OK] File riorganizzati correttamente!

Struttura finale pulita per l'allenamento. Classi reali pronte: ['cartella_pentola', 'pericolo', 'no_pericolo']


In [14]:
import os
from PIL import Image

data_dir = '/content/cartella_estratta'

# Estensioni permesse
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

file_rimossi_estensione = 0
file_rimossi_corrotti = 0

print("Inizio pulizia approfondita della cartella...")

# Ciclo per rimuovere i file non validi o corrotti
for root, dirs, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)

        # 1. Controllo Estensione: Se il nome è sbagliato, elimina subito
        if not file.lower().endswith(valid_extensions):
            os.remove(file_path)
            print(f"Rimosso (Estensione non valida): {file}")
            file_rimossi_estensione += 1
            continue # Salta il resto e passa al prossimo file

        # 2. Controllo Integrità: Proviamo ad aprire l'immagine per vedere se è rotta
        try:
            # Verifica l'intestazione del file
            with Image.open(file_path) as img:
                img.verify()

            # Forza la lettura dei pixel (scova i file parzialmente scaricati)
            with Image.open(file_path) as img:
                img.resize((10, 10))

        except Exception as e:
            # Se PIL va in errore, il file è finto o corrotto: eliminiamolo
            os.remove(file_path)
            print(f"Rimosso (Immagine corrotta/illeggibile): {file}")
            file_rimossi_corrotti += 1

print("\n--- RESOCONTO PULIZIA ---")
print(f"File rimossi per estensione errata: {file_rimossi_estensione}")
print(f"File rimossi perché corrotti:       {file_rimossi_corrotti}")
print("Pulizia completata! Ora puoi caricare il dataset in TensorFlow in totale sicurezza.")

Inizio pulizia approfondita della cartella...

--- RESOCONTO PULIZIA ---
File rimossi per estensione errata: 0
File rimossi perché corrotti:       0
Pulizia completata! Ora puoi caricare il dataset in TensorFlow in totale sicurezza.


In [17]:
import tensorflow as tf
from tensorflow.keras import layers, models

# ==========================================
# 1. CARICAMENTO DATI (Rilevazione automatica 3 classi)
# ==========================================
IMG_SIZE = (64, 64)  # Mantenuto a 64x64 come nel tuo progetto originale
BATCH_SIZE = 32
data_dir = '/content/cartella_estratta'

print("--- 1. Caricamento Dataset in Corso ---")
try:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'  # Configurato per 3 classi
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'  # Configurato per 3 classi
    )

    print(f"\n[OK] Classi rilevate correttamente: {train_ds.class_names}")

except Exception as e:
    print(f"\n❌ ERRORE NEL CARICAMENTO: Controlla che la cartella '{data_dir}' esista e contenga le sottocartelle delle classi.")
    raise e

# ==========================================
# 2. CREAZIONE DELLA RETE NEURALE
# ==========================================
print("\n--- 2. Creazione della Struttura del Modello ---")
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    # 3 uscite con softmax per gestire le probabilità delle 3 classi
    layers.Dense(3, activation='softmax')
])

rilevatore_multiclasse = model

# ==========================================
# 3. COMPILAZIONE
# ==========================================
rilevatore_multiclasse.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Specifica per 3 classi
    metrics=['accuracy']
)

rilevatore_multiclasse.summary()

# ==========================================
# 4. ALLENAMENTO
# ==========================================
print("\n--- 3. Inizio Allenamento del Rilevatore ---")
history = rilevatore_multiclasse.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

# ==========================================
# 5. SALVATAGGIO
# ==========================================
rilevatore_multiclasse.save('modello_3_classi.h5')
print("\n[FINITO] Modello salvato con successo come 'modello_3_classi.h5'")

--- 1. Caricamento Dataset in Corso ---
Found 2841 files belonging to 3 classes.
Using 2273 files for training.
Found 2841 files belonging to 3 classes.
Using 568 files for validation.

[OK] Classi rilevate correttamente: ['cartella_pentola', 'no_pericolo', 'pericolo']

--- 2. Creazione della Struttura del Modello ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       524,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529,635 (2.02 MB)

 Trainable params: 529,635 (2.02 MB)

 Non-trainable params: 0 (0.00 B)


--- 3. Inizio Allenamento del Rilevatore ---
Epoch 1/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 15s 178ms/step - accuracy: 0.9617 - loss: 0.1009 - val_accuracy: 1.0000 - val_loss: 5.0073e-04
Epoch 2/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 13s 184ms/step - accuracy: 1.0000 - loss: 2.2893e-04 - val_accuracy: 1.0000 - val_loss: 1.2352e-04
Epoch 3/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 15s 205ms/step - accuracy: 1.0000 - loss: 1.2043e-04 - val_accuracy: 1.0000 - val_loss: 7.6622e-05
Epoch 4/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 19s 189ms/step - accuracy: 1.0000 - loss: 7.8243e-05 - val_accuracy: 1.0000 - val_loss: 5.7228e-05
Epoch 5/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 21s 201ms/step - accuracy: 1.0000 - loss: 5.5256e-05 - val_accuracy: 1.0000 - val_loss: 4.1935e-05
Epoch 6/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 13s 178ms/step - accuracy: 1.0000 - loss: 4.1095e-05 - val_accuracy: 1.0000 - val_loss: 3.0291e-05
Epoch 7/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 187ms/step - accuracy: 1.0000 - loss: 3.1247e-05 - val_accuracy: 1.0000 - val_loss: 2.4776e-05


[FINITO] Modello salvato con successo come 'modello_3_classi.h5'


In [19]:
import tensorflow as tf
import os

# 1. Configurazione dei percorsi
keras_model_path = 'modello_3_classi.h5'
tflite_quant_model_path = 'modello_3_classi_quant.tflite'

print("--- Inizio Processo di Conversione e Quantizzazione INT8 ---")

if not os.path.exists(keras_model_path):
    raise FileNotFoundError(f"Errore: Il file '{keras_model_path}' non esiste. Assicurati di aver completato l'addestramento.")

# 2. Caricamento del modello Keras a 3 classi
model = tf.keras.models.load_model(keras_model_path)

# 3. Creazione del convertitore TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 4. Definizione del generatore di dati rappresentativi (Necessario per INT8)
def representative_data_gen():
    # Prende un batch dal dataset di addestramento esistente (creato nella cella precedente)
    for input_value, _ in train_ds.take(1):
        # Genera i campioni uno alla volta trasformandoli nel tipo richiesto
        for img in input_value:
            # Espande le dimensioni per simulare un batch di 1 singola immagine (1, 64, 64, 3)
            yield [tf.expand_dims(img, axis=0)]

# 5. Configurazione della quantizzazione INT8 "Full Integer"
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8   # Converte l'input della fotocamera in INT8
converter.inference_output_type = tf.int8  # Restituisce l'output delle 3 classi in INT8

# 6. Conversione effettiva
print("Conversione in corso (Generazione dei pesi INT8)...")
tflite_quant_model = converter.convert()

# 7. Salvataggio del file finale per il microcontrollore
with open(tflite_quant_model_path, 'wb') as f:
    f.write(tflite_quant_model)

# 8. Calcolo e stampa del resoconto sulle dimensioni finali
size_h5 = os.path.getsize(keras_model_path) / 1024
size_tflite = os.path.getsize(tflite_quant_model_path) / 1024

print("\n" + "="*50)
print("🎉 CONVERSIONE COMPLETATA CON SUCCESSO!")
print("="*50)
print(f"📦 Modello Keras Originale (.h5):      {size_h5:.2f} KB")
print(f"⚡ Modello TFLite Quantizzato (INT8): {size_tflite:.2f} KB")
print("="*50)
print(f"👉 Il file '{tflite_quant_model_path}' è pronto! Scaricalo dal menu laterale di Colab.")

--- Inizio Processo di Conversione e Quantizzazione INT8 ---
Conversione in corso (Generazione dei pesi INT8)...
Saved artifact at '/tmp/tmp3ztwc17p'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  132631992027856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132631992028240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132631992041296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132629686896592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132629686901968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132629686902352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132629686900816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132629686901008: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



🎉 CONVERSIONE COMPLETATA CON SUCCESSO!
📦 Modello Keras Originale (.h5):      6246.52 KB
⚡ Modello TFLite Quantizzato (INT8): 524.66 KB
👉 Il file 'modello_3_classi_quant.tflite' è pronto! Scaricalo dal menu laterale di Colab.
